# Proyecto de Regresión: Predicción de Nota Final G3 (Dataset UCI student-mat.csv)

Este notebook implementa el pipeline completo de machine learning para predecir la nota final (`G3`, en escala de 0 a 20 puntos) de estudiantes a partir de su nota del primer parcial (`G1`) y variables conductuales/demográficas.

### Prevención de Data Leakage:
Se elimina explícitamente la variable `G2` (nota del segundo parcial) de las predictoras $X$ para garantizar una predicción temprana genuina que sirva como herramienta preventiva.

## 1. Importación de Librerías

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42


## 2. Carga del Dataset de UCI y Definición del Target

In [ ]:
# 1. Cargar el dataset de UCI (student-mat.csv con sep=';')
path_dataset = 'dataset/student-mat.csv' if os.path.exists('dataset/student-mat.csv') else 'student-mat.csv'
df = pd.read_csv(path_dataset, sep=';')
print('Dimensiones del dataset original:', df.shape)
df.head()


## 3. Prevención de Leakage (Eliminación de G2 y G3 de X)

In [ ]:
# 2. Reconfiguración: Definir Target y Evitar Leakage
TARGET = 'G3'

# Eliminamos G2 y G3 de las predictoras X para mantener solo G1 y hábitos/demografía
X_full = df.drop(columns=['G2', 'G3'])
y_full = df[TARGET]

print('Target:', TARGET, '(Escala 0-20)')
print('Variables predictoras en X_full:', X_full.shape[1])


## 4. Codificación Automática (pd.get_dummies) y División Train/Test

In [ ]:
# 3. Codificación Automática con pd.get_dummies
cat_cols = X_full.select_dtypes(include=['object', 'string']).columns.tolist()
X_encoded = pd.get_dummies(X_full, columns=cat_cols, drop_first=True)
print('Dimensiones tras pd.get_dummies:', X_encoded.shape)

# 4. División Train/Test (80% / 20%)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_encoded, y_full, test_size=0.2, random_state=RANDOM_STATE
)
print(f'Muestras Train: {X_train_raw.shape[0]} | Muestras Test: {X_test_raw.shape[0]}')


## 5. Análisis Estadístico y Selección de Caracterizaciones por Correlación

In [ ]:
# 5. Matriz de Correlación con G3 en el set de Entrenamiento
train_con_target = X_train_raw.copy()
train_con_target[TARGET] = y_train.values

corr_con_target = train_con_target.corr(numeric_only=True)[TARGET].drop(TARGET).sort_values(key=lambda s: s.abs(), ascending=False)
print('--- Correlaciones con G3 (Nota Final) ---')
print(corr_con_target.head(15))


In [ ]:
# 6. Selección de Features (Umbral |r| >= 0.10)
UMBRAL_CORR = 0.10
FEATURES_FINALES = corr_con_target[corr_con_target.abs() >= UMBRAL_CORR].index.tolist()

print(f'Se seleccionaron {len(FEATURES_FINALES)} variables con correlación >= {UMBRAL_CORR}:')
print(FEATURES_FINALES)

X_train = X_train_raw[FEATURES_FINALES].copy()
X_test = X_test_raw[FEATURES_FINALES].copy()


## 6. Escalado StandardScaler y Entrenamiento de Modelos

In [ ]:
# Escalado de variables
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

# Modelo Lineal
modelo_lineal = LinearRegression()
modelo_lineal.fit(X_train_scaled, y_train)

# Modelo Polinomial + Regularización
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

param_grid = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}
grid_ridge = GridSearchCV(Ridge(random_state=RANDOM_STATE), param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=1)
grid_ridge.fit(X_train_poly, y_train)

grid_lasso = GridSearchCV(Lasso(random_state=RANDOM_STATE, max_iter=10000), param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=1)
grid_lasso.fit(X_train_poly, y_train)

if -grid_ridge.best_score_ <= -grid_lasso.best_score_:
    modelo_poly = grid_ridge.best_estimator_
    nombre_regularizador = f'Ridge (alpha={grid_ridge.best_params_["alpha"]})'
else:
    modelo_poly = grid_lasso.best_estimator_
    nombre_regularizador = f'Lasso (alpha={grid_lasso.best_params_["alpha"]})'


## 7. Evaluación Comparativa y Guardado de Artefactos

In [ ]:
def metricas(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
    }

resultados = {
    ('Lineal', 'Train'): metricas(y_train, modelo_lineal.predict(X_train_scaled)),
    ('Lineal', 'Test'):  metricas(y_test,  modelo_lineal.predict(X_test_scaled)),
    ('Polinomial(2)+Reg', 'Train'): metricas(y_train, modelo_poly.predict(X_train_poly)),
    ('Polinomial(2)+Reg', 'Test'):  metricas(y_test,  modelo_poly.predict(X_test_poly)),
}

tabla_comparativa = pd.DataFrame(resultados).T
tabla_comparativa.index.names = ['Modelo', 'Set']
print(tabla_comparativa.round(4))


In [ ]:
rmse_test_lineal = tabla_comparativa.loc[('Lineal', 'Test'), 'RMSE']
rmse_test_poly = tabla_comparativa.loc[('Polinomial(2)+Reg', 'Test'), 'RMSE']

if rmse_test_lineal <= rmse_test_poly:
    mejor_modelo = modelo_lineal
    usa_poly = False
    nombre_mejor = 'LinearRegression'
    mae_test_mejor = tabla_comparativa.loc[('Lineal', 'Test'), 'MAE']
else:
    mejor_modelo = modelo_poly
    usa_poly = True
    nombre_mejor = f'PolynomialFeatures(2) + {nombre_regularizador}'
    mae_test_mejor = tabla_comparativa.loc[('Polinomial(2)+Reg', 'Test'), 'MAE']

print(f'Mejor modelo (menor RMSE en test): {nombre_mejor}')
print(f'RMSE Test: {min(rmse_test_lineal, rmse_test_poly):.4f}  |  MAE Test: {mae_test_mejor:.4f}')

joblib.dump(mejor_modelo, 'total_score_model.pkl')
joblib.dump(scaler, 'total_score_scaler.pkl')
joblib.dump(poly if usa_poly else None, 'total_score_poly.pkl')
joblib.dump(FEATURES_FINALES, 'total_score_features.pkl')
joblib.dump(float(mae_test_mejor), 'total_score_mae.pkl')

print('\nArtefactos guardados exitosamente: total_score_model.pkl, total_score_scaler.pkl,')
print('total_score_poly.pkl, total_score_features.pkl, total_score_mae.pkl')
